# Savant full - averaged replicate labels, regressor only

A controlled variant of `train_savant_full_w_classifier.ipynb`. Two changes, nothing else:

1. **Replicate labels are averaged.** ~11.9% of unique `seq`+`smiles` pairs carry
   more than one measurement, disagreeing by a pooled 0.81 pKd. Training on them
   as separate rows asks the model to fit two different answers to byte-identical
   features, which is pure irreducible error and a direct cause of the prediction
   shrinkage (`pred_std_ratio` ~0.40) seen in the previous run.
2. **The classifier is dropped.** On the benchmark the trained classifier matched
   the regressor at the pKd 6 cut (ROC 0.679 vs 0.674) and lost at pKd 7 (0.559
   vs 0.574). A regressor gives every threshold for free, chosen at inference
   time; a classifier gives one, fixed at training time, no better. It was
   costing half the training time and half the tuning budget to add nothing.

## This reuses the cached features - it does not re-encode anything

The gate, the input file and the unique ligand/protein sets are identical to the
previous run, so the ChemBERTa, ESM2, RDKit and ECFP blocks are byte-for-byte the
same. This notebook reads them from `work_full_v2` and writes only rows and models
to `work_full_v3`.

It reuses `work_full_v2/ligands.csv` and `proteins.csv` **directly** rather than
rebuilding the indices, which guarantees the row indices still line up with the
cached `.npy` blocks. Rebuilding them would risk a silent misalignment that no
width check could catch.

It also reuses the previous run's tuned hyperparameters by default. That is
deliberate: reusing them isolates the label change as the only difference, so a
metric moving is attributable to the averaging. Re-tuning would confound the two.

## Read this before comparing the numbers

**Averaging makes the test labels cleaner, so test RMSE drops partly because the
target got easier - not only because the model got better.** Comparing this run's
RMSE against the previous run's 1.3013 is not apples-to-apples.

The cell that reports metrics therefore also reports them on the subset of test
rows with **exactly one measurement**. Those labels are untouched by averaging,
so that subset *is* directly comparable to the previous run. Use it for the
before/after claim; use the full-test number to describe this model on its own.

In [ ]:
import gc, json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

HERE     = Path.cwd()                       # run from ML_model/
FEATURES = HERE / "work_full_v2"            # cached blocks, read-only here
OUT      = HERE / "work_full_v3"            # rows + models for this run

VARIANTS = {
    "esm650": {"model": "facebook/esm2_t33_650M_UR50D", "file": "protein_esm2_650M.npy",
               "dim": 1280},
    "esm8":   {"model": "facebook/esm2_t6_8M_UR50D",    "file": "protein_esm2_8M.npy",
               "dim": 320},
}

SEED       = 1337
BINDER_PKD = 6.0            # kept only so the bundle records savant4's threshold
N_THREADS  = -1
TUNE_ROWS, TUNE_ITERS = 200_000, 12

## Weighting a row by how many measurements its label averages is statistically
## principled (the mean of n measurements has variance sigma^2/n), but it changes
## two things at once and makes this run no longer a clean test of averaging
## alone. Off by default; turn it on for a separate run.
SAMPLE_WEIGHT_BY_N = False

OUT.mkdir(parents=True, exist_ok=True)
def log(m): print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)

NEED = ["dataset_rows.csv", "ligands.csv", "proteins.csv", "rdkit_columns.json",
        "ligand_chemberta.npy", "ligand_rdkit.npy", "ligand_ecfp.npy",
        "protein_aa.npy"] + [c["file"] for c in VARIANTS.values()]
missing = [f for f in NEED if not (FEATURES / f).exists()]
if missing:
    raise SystemExit(
        f"{FEATURES} is missing {missing}.\n"
        "This notebook reuses the feature blocks built by train_savant_full_w_classifier.ipynb "
        "rather than re-encoding ~1M ligands and ~23k proteins. Run that notebook "
        "first, or point FEATURES at a directory that has them."
    )
log(f"features {FEATURES}")
log(f"out      {OUT}")

## 1. Average the replicates

Grouping on the integer ligand/protein ids, not on the raw `seq` and `smiles`
strings - some sequences run to several thousand characters and hashing them for
a 1.8M-row groupby is far slower for an identical result.

`n_measurements` and `label_sd` are kept per row. They are the diagnostic for how
much this changed, and `n_measurements` is what `SAMPLE_WEIGHT_BY_N` would use.

In [ ]:
raw = pd.read_csv(FEATURES / "dataset_rows.csv")
log(f"{len(raw):,} rows in, {raw.ligand_id.nunique():,} ligands, "
    f"{raw.protein_id.nunique():,} proteins")

g = raw.groupby(["ligand_id", "protein_id"], sort=False)["neg_log10_affinity_M"]
rows_all = g.agg(neg_log10_affinity_M="mean", n_measurements="size",
                 label_sd="std").reset_index()
rows_all["label_sd"] = rows_all.label_sd.fillna(0.0)

dup = rows_all[rows_all.n_measurements > 1]
print(f"rows {len(raw):,} -> {len(rows_all):,}  "
      f"({len(raw) - len(rows_all):,} collapsed, {100*(1-len(rows_all)/len(raw)):.1f}%)")
print(f"pairs with >1 measurement: {len(dup):,} ({100*len(dup)/len(rows_all):.1f}%)")
print(f"  their disagreement (sd):  median {dup.label_sd.median():.3f}  "
      f"pooled {np.sqrt((dup.label_sd**2).mean()):.3f} pKd")
print(f"  max measurements on one pair: {rows_all.n_measurements.max()}")
print(f"\nlabel sd: before {raw.neg_log10_affinity_M.std():.4f}  "
      f"after {rows_all.neg_log10_affinity_M.std():.4f}")
rows_all.to_csv(OUT / "dataset_rows.csv", index=False)
del raw, g; gc.collect()

## 2. Split

Protein-grouped with the same seed as the previous run, and the inner
early-stopping split holds out the same key as the outer one.

The seed matches but the **split will not be identical** - there are fewer rows
now, so `GroupShuffleSplit` lands differently. It is still the same *kind* of
split on the same protein pool, which is what the comparison needs.

Rows are reordered to `fit | val | test` before the matrix is built so each split
is a contiguous view rather than a copy.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

y_all  = rows_all.neg_log10_affinity_M.to_numpy(np.float32)
groups = rows_all.protein_id.to_numpy()

tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED)
              .split(np.zeros(len(rows_all)), y_all, groups))
i_fit, i_val = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED)
                    .split(tr, y_all[tr], groups=groups[tr]))
fit, val = tr[i_fit], tr[i_val]
N_FIT, N_VAL, N_TE = len(fit), len(val), len(te)
CUT = N_FIT + N_VAL

rows = rows_all.iloc[np.concatenate([fit, val, te])].reset_index(drop=True)
y     = rows.neg_log10_affinity_M.to_numpy(np.float32)
n_obs = rows.n_measurements.to_numpy(np.float32)

assert not (set(groups[fit]) & set(groups[te])), "protein leaked fit -> test"
assert not (set(groups[val]) & set(groups[te])), "protein leaked val -> test"
print(f"fit {N_FIT:,}  val {N_VAL:,}  test {N_TE:,}")
print(f"constant-mean RMSE  val {y[N_FIT:CUT].std():.4f}  test {y[CUT:].std():.4f}")
print(f"  (a model above its own line here is not beating 'predict the average')")
single = n_obs[CUT:] == 1
print(f"test rows with exactly 1 measurement: {int(single.sum()):,} "
      f"({100*single.mean():.1f}%) - the subset comparable to the previous run")

In [ ]:
def assemble_variant(rows, esm_file, chunk=200_000):
    '''(n, width) float32 in the training block order, from the cached blocks.'''
    lparts = [np.load(FEATURES / f, mmap_mode="r") for f in
              ("ligand_chemberta.npy", "ligand_rdkit.npy", "ligand_ecfp.npy")]
    pparts = [np.load(FEATURES / f, mmap_mode="r") for f in (esm_file, "protein_aa.npy")]
    width  = sum(p.shape[1] for p in lparts + pparts)

    X = np.empty((len(rows), width), dtype=np.float32)
    log(f"matrix {X.shape} ({X.nbytes / 1e9:.1f} GB) from {esm_file}")
    li, pi = rows.ligand_id.to_numpy(), rows.protein_id.to_numpy()
    for start in range(0, len(rows), chunk):
        stop, col = min(start + chunk, len(rows)), 0
        for part, ids in ((lparts, li), (pparts, pi)):
            for block in part:
                X[start:stop, col:col + block.shape[1]] = block[ids[start:stop]]
                col += block.shape[1]
    return X

RD_COLS = json.loads((FEATURES / "rdkit_columns.json").read_text())
for name, cfg in VARIANTS.items():
    cfg["width"] = 384 + len(RD_COLS) + 1024 + cfg["dim"] + 21
    print(f"  {name:8s} {cfg['width']} features")
## the cached blocks must still describe the ids we just grouped on
assert rows_all.ligand_id.max() < np.load(FEATURES / "ligand_ecfp.npy", mmap_mode="r").shape[0]
assert rows_all.protein_id.max() < np.load(FEATURES / "protein_aa.npy", mmap_mode="r").shape[0]
print("ligand/protein ids are within the cached blocks")

## 3. Hyperparameters

Reused from the previous run by default, so the label averaging is the only
difference between the two. Delete `best_params_reg.json` from `OUT` and set
`RETUNE = True` to search again - but then the comparison measures both changes
together, not the averaging alone.

In [ ]:
import xgboost as xgb

RETUNE = False
param_file = OUT / "best_params_reg.json"
prev = FEATURES / "best_params_reg.json"

if param_file.exists() and not RETUNE:
    BEST_REG = json.loads(param_file.read_text())
    log(f"reusing {param_file}: {BEST_REG}")
elif prev.exists() and not RETUNE:
    BEST_REG = json.loads(prev.read_text())
    param_file.write_text(json.dumps(BEST_REG, indent=1))
    log(f"copied params from the previous run: {BEST_REG}")
else:
    rng = np.random.default_rng(SEED)
    SPACE = {"max_depth": [6, 7, 8, 9, 10, 12], "eta": [0.03, 0.05, 0.08, 0.12],
             "subsample": [0.6, 0.7, 0.8, 0.9], "colsample_bytree": [0.3, 0.4, 0.5, 0.7],
             "min_child_weight": [1, 3, 5, 10, 20], "reg_lambda": [0.5, 1.0, 2.0, 5.0]}
    sub = np.sort(rng.choice(N_FIT, size=min(TUNE_ROWS, N_FIT), replace=False))
    Xt = assemble_variant(rows.iloc[sub], VARIANTS["esm650"]["file"])
    Xv = assemble_variant(rows.iloc[N_FIT:CUT], VARIANTS["esm650"]["file"])
    dt = xgb.QuantileDMatrix(Xt, label=y[sub])
    dv = xgb.QuantileDMatrix(Xv, label=y[N_FIT:CUT], ref=dt)
    del Xt, Xv; gc.collect()

    trials = []
    for i in range(TUNE_ITERS):
        p = {k: v[rng.integers(len(v))] for k, v in SPACE.items()}
        b = xgb.train({**p, "objective": "reg:squarederror", "eval_metric": "rmse",
                       "tree_method": "hist", "nthread": N_THREADS, "seed": SEED},
                      dt, num_boost_round=1500, evals=[(dv, "val")],
                      early_stopping_rounds=40, verbose_eval=False)
        trials.append({**p, "rmse": float(b.best_score), "best_iter": int(b.best_iteration)})
        pd.DataFrame(trials).to_csv(OUT / "tuning_trials_reg.csv", index=False)
        log(f"  [{i+1}/{TUNE_ITERS}] rmse {b.best_score:.4f} @{b.best_iteration}  {p}")
    t = pd.DataFrame(trials).sort_values("rmse").reset_index(drop=True)
    display(t)
    ## .item(): pandas hands back np.int64, which json.dumps refuses
    BEST_REG = {k: (t.loc[0, k].item() if hasattr(t.loc[0, k], "item") else t.loc[0, k])
                for k in SPACE}
    param_file.write_text(json.dumps(BEST_REG, indent=1))
    del dt, dv; gc.collect()
    log(f"best rmse {t.loc[0, 'rmse']:.4f}  {BEST_REG}")

## 4. Train

One regressor per variant, one variant at a time. No classifier.

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def block_metrics(yt, pred, tag):
    err = pred - yt
    return {f"{tag}_n": int(len(yt)),
            f"{tag}_RMSE": float(np.sqrt((err ** 2).mean())),
            f"{tag}_MAE": float(np.abs(err).mean()),
            f"{tag}_R2": float(r2_score(yt, pred)),
            f"{tag}_Pearson": float(np.corrcoef(yt, pred)[0, 1]),
            f"{tag}_Spearman": float(spearmanr(yt, pred).correlation),
            f"{tag}_pred_std_ratio": float(pred.std() / yt.std()),
            f"{tag}_const_RMSE": float(yt.std())}

RESULTS = {}
for name, cfg in VARIANTS.items():
    log(f"===== {name} ({cfg['width']} features) =====")
    X = assemble_variant(rows, cfg["file"])

    w = n_obs[:N_FIT] if SAMPLE_WEIGHT_BY_N else None
    dfit = xgb.QuantileDMatrix(X[:N_FIT], label=y[:N_FIT], weight=w)
    dval = xgb.QuantileDMatrix(X[N_FIT:CUT], label=y[N_FIT:CUT], ref=dfit)
    reg = xgb.train({**BEST_REG, "objective": "reg:squarederror", "eval_metric": "rmse",
                     "tree_method": "hist", "nthread": N_THREADS, "seed": SEED},
                    dfit, num_boost_round=3000, evals=[(dval, "val")],
                    early_stopping_rounds=50, verbose_eval=200)
    pred = reg.inplace_predict(X[CUT:])
    del dfit, dval

    yt = y[CUT:]
    r = {"variant": name, "n_features": int(cfg["width"]), "esm_model": cfg["model"],
         "esm_dim": cfg["dim"], "n_fit": N_FIT, "n_val": N_VAL, "n_test": N_TE,
         "reg_best_iteration": int(reg.best_iteration),
         "sample_weighted": bool(SAMPLE_WEIGHT_BY_N),
         "averaged_labels": True, "has_classifier": False}
    r.update(block_metrics(yt, pred, "test"))
    ## labels untouched by averaging -> directly comparable to the previous run
    r.update(block_metrics(yt[single], pred[single], "single"))
    RESULTS[name] = r
    for k, v in r.items():
        print(f"  {k:22s} {v}")

    vdir = OUT / name
    vdir.mkdir(exist_ok=True)
    reg.save_model(vdir / "xgb_regressor.json")
    json.dump(r, open(vdir / "metrics.json", "w"), indent=1)
    log(f"saved {vdir}/xgb_regressor.json, metrics.json")
    del X, reg; gc.collect()

## 5. Results

Two blocks per variant:

- **`test_*`** - the whole test set against averaged labels. This describes the
  model as it now is.
- **`single_*`** - only test rows with one measurement, whose labels averaging did
  not touch. **This is the row to compare against the previous run.**

If `test_RMSE` improves but `single_RMSE` does not, the gain was the target
getting cleaner rather than the model getting better. That is still a real
improvement in label quality, but it is a different claim and should be worded
as one.

In [ ]:
res = pd.DataFrame(RESULTS).T
show = ["n_features", "test_n", "test_RMSE", "test_const_RMSE", "test_R2",
        "test_Pearson", "test_Spearman", "test_pred_std_ratio",
        "single_n", "single_RMSE", "single_const_RMSE", "single_Pearson",
        "single_pred_std_ratio"]
display(res[show].astype(float).round(4))
res.to_csv(OUT / "variant_comparison.csv")

print("\nprevious run (train_savant_full_w_classifier.ipynb), for reference:")
for name in VARIANTS:
    p = FEATURES / name / "metrics.json"
    if p.exists():
        m = json.loads(p.read_text())
        print(f"  {name:8s} RMSE {m['RMSE']:.4f}  Pearson {m['Pearson']:.4f}  "
              f"pred_std_ratio {m['pred_std_ratio']:.4f}")
    else:
        print(f"  {name:8s} no metrics.json in {FEATURES / name}")
print("\nCompare those against single_* above, not against test_*.")

## 6. Bundles

Regressor only. `savant1.py` treats `classifier_json` as optional: with no
classifier it returns `p_binder` as NaN and derives `is_binder` by thresholding
the predicted pKd at `binder_pkd`, so the output schema is unchanged and the
scoring notebook needs no edits.

In [ ]:
import joblib

for name, cfg in VARIANTS.items():
    vdir = OUT / name
    bundle = {
        "variant": name,
        "esm_model": cfg["model"],
        "block_order": ["chemberta", "rdkit_named", "ecfp", "esm", "aa"],
        "block_widths": [384, len(RD_COLS), 1024, cfg["dim"], 21],
        "n_features": int(cfg["width"]),
        "rdkit_columns": RD_COLS,
        "regressor_json": (vdir / "xgb_regressor.json").read_bytes(),
        "classifier_json": None,            # dropped on purpose
        "metrics": RESULTS[name],
        "xgboost_version": xgb.__version__,
        "binder_pkd": BINDER_PKD,
        "averaged_labels": True,
    }
    assert sum(bundle["block_widths"]) == bundle["n_features"]
    path = OUT / f"savant_full_{name}.pkl"
    joblib.dump(bundle, path, compress=3)
    print(f"{path.name}  {path.stat().st_size / 1e6:.0f} MB  "
          f"{bundle['n_features']} features  regressor only")

print(f"\nTo score these, point the notebook's WORK at {OUT}")
print("   WORK = REPO / 'ML_model' / 'work_full_v3'")